# Eksplorasi Fenomena Dinamika Klinis COVID-19 dan Pengumpulan Data
**Mata Kuliah:** Pengantar Proses Stokastik  
**Dosen Pengampu:** Ade Susanti, S.Si., M.Si.  
**Kelompok 1:**
- Achika Vigo Azhyra (M0125001)
- Fadhila Hardi Ningrum (M0124004)
- Fawwaz Absyar Rifai (M0125044)
- Karunia Febyayu Puspitaningtyas (M0124010)
- Ramadhan Imanur Rochim (M0124015)

---

## 1. Latar Belakang dan Urgensi Fenomena

Pandemi *Coronavirus Disease 2019* (COVID-19) yang melanda Indonesia sejak Maret 2020 hingga akhir 2022 merupakan salah satu peristiwa kesehatan masyarakat terbesar dalam sejarah modern. Dinamika perkembangan klinis seorang pasien terkonfirmasi positif memiliki karakteristik alami ketidakpastian (*stochastic nature*) yang bergerak melintasi berbagai tahapan keparahan klinis sebelum akhirnya berujung pada salah satu dari dua kondisi permanen: **Sembuh** atau **Meninggal Dunia**.

Dalam pemodelan proses stokastik, tahapan klinis aktif merepresentasikan **keadaan transien** (*transient state*), sedangkan status kesembuhan dan kematian merupakan **keadaan penyerap** (*absorbing state*) karena begitu seorang individu mencapai keadaan tersebut, ia keluar dari dinamika episode klinis aktif tersebut.

Notebook ini bertujuan untuk:
1. Memuat dan mengeksplorasi data empiris agregat COVID-19 Indonesia dari Satgas Penanganan COVID-19 / BNPB.
2. Menganalisis parameter epidemiologi kunci (rasio fatalitas kasus/CFR, rasio kesembuhan/CRR, dan rata-rata durasi kasus aktif).
3. Mendefinisikan ruang keadaan klinis berbasis pedoman tata laksana klinis Kementerian Kesehatan Republik Indonesia (Kemenkes RI) dan WHO.
4. Menyusun matriks peluang transisi satu langkah $P$ dan memverifikasi syarat mutlak matriks stokastik.


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 120

sys.path.append(os.path.abspath("../scripts"))
import fsa_solver as fsa

print("Seluruh pustaka dan modul pendukung berhasil dimuat.")


Seluruh pustaka dan modul pendukung berhasil dimuat.


## 2. Pemuatan dan Eksplorasi Data Empiris COVID-19 Indonesia

Data yang digunakan merupakan data runtun waktu resmi penanganan COVID-19 di Indonesia (`covid_19_indonesia_time_series_all.csv`) yang mencakup periode awal pandemi (Maret 2020) hingga September 2022.


In [2]:
raw_data_path = "../data/raw/covid_19_indonesia_time_series_all.csv"
df_raw = pd.read_csv(raw_data_path)
print(f"Dimensi data mentah: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")

df_indo = df_raw[df_raw["Location"] == "Indonesia"].copy()
df_indo["Date"] = pd.to_datetime(df_indo["Date"])
df_indo = df_indo.sort_values("Date").reset_index(drop=True)

display(df_indo[["Date", "New Cases", "New Deaths", "New Recovered", "Total Cases", "Total Deaths", "Total Recovered", "Total Active Cases"]].tail())


Dimensi data mentah: 31822 baris, 38 kolom
          Date  New Cases  ...  Total Recovered  Total Active Cases
924 2022-09-12       1848  ...          6204241               32312
925 2022-09-13       2896  ...          6207858               31571
926 2022-09-14       2799  ...          6211796               30411
927 2022-09-15       2651  ...          6215711               29126
928 2022-09-16       2358  ...          6218708               28460

[5 rows x 8 columns]


### Ringkasan Statistik Kumulatif Nasional

Berdasarkan status kumulatif per 9 September 2022, diperoleh gambaran makro keluaran (*outcome*) epidemiologis pasien di Indonesia:


In [3]:
latest = df_indo.iloc[-1]
total_cases = latest["Total Cases"]
total_recov = latest["Total Recovered"]
total_death = latest["Total Deaths"]
total_active = latest["Total Active Cases"]

cfr = (total_death / total_cases) * 100
crr = (total_recov / total_cases) * 100

print("=" * 50)
print("RINGKASAN STATUS EPIDEMIOLOGI COVID-19 INDONESIA")
print("=" * 50)
print(f"Total Kasus Terkonfirmasi : {total_cases:,}")
print(f"Total Pasien Sembuh       : {total_recov:,} ({crr:.2f}%)")
print(f"Total Pasien Meninggal    : {total_death:,} ({cfr:.2f}%)")
print(f"Total Kasus Aktif Terakhir: {total_active:,} ({(total_active/total_cases)*100:.2f}%)")
print("=" * 50)


RINGKASAN STATUS EPIDEMIOLOGI COVID-19 INDONESIA
Total Kasus Terkonfirmasi : 6,405,044
Total Pasien Sembuh       : 6,218,708 (97.09%)
Total Pasien Meninggal    : 157,876 (2.46%)
Total Kasus Aktif Terakhir: 28,460 (0.44%)


### Estimasi Durasi Kasus Aktif

Untuk memahami parameter waktu penyerapan, dianalisis laju harian pelepasan status aktif (sembuh dan meninggal) terhadap populasi aktif hari sebelumnya:
$$\\lambda_{\\text{exit}} = \\frac{\\text{New Recovered}_t + \\text{New Deaths}_t}{\\text{Active Cases}_{t-1}}$$
sehingga ekspektasi durasi penyelesaian episode klinis diperkirakan melalui $\\mathbb{E}[D] = 1 / \\lambda_{\\text{exit}}$.


In [4]:
df_indo["Prev_Active"] = df_indo["Total Active Cases"].shift(1)
valid_active = df_indo[df_indo["Prev_Active"] > 5000].copy()

valid_active["Daily_Exit_Rate"] = (valid_active["New Recovered"] + valid_active["New Deaths"]) / valid_active["Prev_Active"]
mean_exit_rate = valid_active["Daily_Exit_Rate"].mean()
mean_duration_days = 1.0 / mean_exit_rate

print(f"Rata-rata Laju Transisi Keluar Harian : {mean_exit_rate:.4f} per hari")
print(f"Estimasi Rata-rata Durasi Kasus Aktif : {mean_duration_days:.2f} hari (~{mean_duration_days/7:.2f} minggu)")


Rata-rata Laju Transisi Keluar Harian : 0.0656 per hari
Estimasi Rata-rata Durasi Kasus Aktif : 15.24 hari (~2.18 minggu)


## 3. Visualisasi Tren Epidemiologis COVID-19 Indonesia


In [5]:
fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax[0].plot(df_indo["Date"], df_indo["New Cases"], label="Kasus Baru Harian", color="#e74c3c", lw=1.2, alpha=0.8)
ax[0].plot(df_indo["Date"], df_indo["New Recovered"], label="Sembuh Harian", color="#2ecc71", lw=1.2, alpha=0.8)
ax[0].set_title("Dinamika Kasus Harian COVID-19 di Indonesia (2020-2022)", fontsize=12, fontweight="bold")
ax[0].set_ylabel("Jiwa per Hari")
ax[0].legend(loc="upper right")

ax[1].fill_between(df_indo["Date"], df_indo["Total Active Cases"], color="#3498db", alpha=0.4, label="Kasus Aktif (Dalam Perawatan)")
ax[1].plot(df_indo["Date"], df_indo["Total Active Cases"], color="#2980b9", lw=1.5)
ax[1].set_title("Beban Kasus Aktif (Tahap Transien) yang Memerlukan Triase Klinis", fontsize=12, fontweight="bold")
ax[1].set_xlabel("Tanggal")
ax[1].set_ylabel("Jumlah Kasus Aktif")
ax[1].legend(loc="upper right")

plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/tren_covid_indonesia.png", dpi=300)
plt.close(fig)
print("Grafik tren epidemiologi berhasil disimpan di figures/tren_covid_indonesia.png")


Grafik tren epidemiologi berhasil disimpan di figures/tren_covid_indonesia.png


## 4. Definisi Ruang Keadaan Model dan Matriks Transisi ($P$)

Berdasarkan Buku Pedoman Tata Laksana COVID-19 Kemenkes RI dan konsensus klinis, pasien aktif diklasifikasikan ke dalam 3 keadaan transien dan 2 keadaan penyerap dengan siklus pengamatan mingguan ($\\Delta t = 1$ minggu / 7 hari):

1. **Keadaan Transien ($S_T$):**
   * $T_1$ : Gejala Ringan / Isolasi Mandiri (*Mild / Self-Isolation*)
   * $T_2$ : Gejala Sedang / Rawat Inap Ruang Isolasi Non-ICU (*Moderate Inpatient*)
   * $T_3$ : Gejala Berat-Kritis / Perawatan Intensif (*Severe-Critical / ICU*)
2. **Keadaan Penyerap ($S_A$):**
   * $A_1$ : Sembuh / Selesai Isolasi (*Recovered / Discharged*)
   * $A_2$ : Meninggal Dunia (*Deceased / Fatality*)

Ruang keadaan total: $S = \\{T_1, T_2, T_3, A_1, A_2\\}$.


In [6]:
model = fsa.get_covid_clinical_model()
state_labels = model["state_labels"]
P_matrix = model["P_float"]

df_P = pd.DataFrame(P_matrix, index=state_labels, columns=state_labels)
print("Matriks Peluang Transisi Satu Langkah (P):")
display(df_P)

is_valid, issues = fsa.verify_stochastic_matrix(P_matrix)
print("\n--- Verifikasi Aksioma Matriks Stokastik ---")
print("Status Validitas:", "TERPENUHI (VALID)" if is_valid else f"GAGAL: {issues}")
for idx, row in df_P.iterrows():
    print(f"Baris {idx:<15}: Jumlah = {row.sum():.6f} | Non-negatif = {np.all(row >= 0)}")


Matriks Peluang Transisi Satu Langkah (P):
                Ringan (T1)  Sedang (T2)  ...  Sembuh (A1)  Meninggal (A2)
Ringan (T1)            0.25         0.05  ...         0.69            0.01
Sedang (T2)            0.10         0.30  ...         0.45            0.05
Berat/ICU (T3)         0.00         0.20  ...         0.25            0.30
Sembuh (A1)            0.00         0.00  ...         1.00            0.00
Meninggal (A2)         0.00         0.00  ...         0.00            1.00

[5 rows x 5 columns]

--- Verifikasi Aksioma Matriks Stokastik ---
Status Validitas: TERPENUHI (VALID)
Baris Ringan (T1)    : Jumlah = 1.000000 | Non-negatif = True
Baris Sedang (T2)    : Jumlah = 1.000000 | Non-negatif = True
Baris Berat/ICU (T3) : Jumlah = 1.000000 | Non-negatif = True
Baris Sembuh (A1)    : Jumlah = 1.000000 | Non-negatif = True
Baris Meninggal (A2) : Jumlah = 1.000000 | Non-negatif = True


## 5. Penyimpanan Data Terproses

Matriks peluang transisi dan partisi kanoniknya disimpan ke dalam direktori `data/processed/` untuk dianalisis lebih lanjut pada tahap *First Step Analysis*.


In [7]:
os.makedirs("../data/processed", exist_ok=True)
df_P.to_csv("../data/processed/matriks_transisi.csv")

df_Q = pd.DataFrame(model["P_float"][:3, :3], index=model["transient_labels"], columns=model["transient_labels"])
df_Q.to_csv("../data/processed/matriks_partisi_Q.csv")

df_R = pd.DataFrame(model["P_float"][:3, 3:], index=model["transient_labels"], columns=model["absorbing_labels"])
df_R.to_csv("../data/processed/matriks_partisi_R.csv")

print("Seluruh matriks berhasil disimpan di data/processed/")


Seluruh matriks berhasil disimpan di data/processed/
